I2 portable reader candidate. New outputs are isolated under runs/. Constructed examples demonstrate methods; fixed filing facts/figures are historical teaching inputs. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch03_data_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 3: corporate actions and point-in-time data

The fixture values are constructed teaching data, not actual issuers or market observations. The permanent `security_id`, reporting period, source availability time, and local ingestion time are kept distinct. A version is visible only when both availability and ingestion have occurred by `signal_time`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from quant_course import pit_asof_join

signals = pd.read_csv(ROOT / "data" / "fixtures" / "a1_pit_signals.csv")
versions = pd.read_csv(ROOT / "data" / "fixtures" / "a1_pit_versions.csv")
signals["signal_time"] = pd.to_datetime(signals["signal_time"], utc=True)
for column in ("available_at", "ingested_at"):
    versions[column] = pd.to_datetime(versions[column], utc=True)
joined = pit_asof_join(signals, versions)
joined[["signal_id", "security_id", "signal_time", "version_id", "eps", "pit_effective_at"]]


In [ ]:
expected = [None, "original_10", "original_10", "original_10", "restated_8", "other_issuer_77"]
actual = joined["version_id"].astype("object").where(joined["version_id"].notna(), None).tolist()
assert actual == expected
eligible = joined[joined["eps"].notna()]
assert (eligible["available_at"] <= eligible["signal_time"]).all()
assert (eligible["ingested_at"] <= eligible["signal_time"]).all()


## Split and dividend

A constructed 2:1 split turns one old share into two new shares. The post-action price is 51 and each new share receives a cash dividend of 1, so wealth is `2 × 51 + 2 × 1 = 104` against an initial price of 100, a 4% total return.

In [ ]:
actions = pd.read_csv(ROOT / "data" / "fixtures" / "a1_corporate_actions.csv")
action = actions.iloc[0]
ending_value = action.split_ratio * action.post_action_price + action.split_ratio * action.new_share_cash_dividend
action_return = ending_value / action.pre_action_price - 1
assert ending_value == 104
assert abs(action_return - 0.04) < 1e-12
print(f"ending wealth={ending_value:g}; total return={action_return:.2%}")


## Daylight-saving boundary

New York skips from 01:59:59 EST to 03:00:00 EDT on 10 March 2024. These two local timestamps are one hour apart in UTC even though the wall clock advances by two hours.

In [ ]:
dst = pd.to_datetime(["2024-03-10T01:30:00-05:00", "2024-03-10T03:30:00-04:00"], utc=True).tz_convert("America/New_York")
elapsed_hours = (dst[1] - dst[0]).total_seconds() / 3600
assert elapsed_hours == 1
print(f"{dst[0]} -> {dst[1]}; elapsed UTC time={elapsed_hours:.0f} hour")


In [ ]:
filing = joined[joined["security_id"] == "SIM-FILING-1"].copy()
x = filing["signal_time"].dt.tz_convert("UTC")
values = filing["eps"].to_numpy(dtype="float64", na_value=float("nan"))
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.step(x, values, where="post", color="#2878B5", linewidth=2, label="PIT-visible EPS")
ax.scatter(x, values, color="#2878B5", zorder=3)
ax.axvline(pd.Timestamp("2024-03-15T20:12:00Z"), color="#E38B2C", linestyle="--", label="Original ingested")
ax.axvline(pd.Timestamp("2024-04-02T20:08:00Z"), color="#D9534F", linestyle="--", label="Restatement ingested")
ax.set_title("Simulated filing: a restatement affects later signals only")
ax.set_xlabel("signal time (UTC)")
ax.set_ylabel("EPS visible at signal time (constructed currency/unit)")
ax.grid(alpha=0.25)
locator = mdates.AutoDateLocator(maxticks=5)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.legend()
fig.autofmt_xdate(rotation=20)
fig.tight_layout()
output = RUN_DIR / "figures" / "a1" / "ch03_timeline.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=160)
plt.close(fig)
output
